In [1]:
!pip show torch transformers

Name: torch
Version: 2.6.0+cu126
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org/
Author: PyTorch Team
Author-email: packages@pytorch.org
License: BSD-3-Clause
Location: C:\Users\yishu\AppData\Local\Programs\Python\Python313\Lib\site-packages
Requires: filelock, fsspec, jinja2, networkx, setuptools, sympy, typing-extensions
Required-by: torchaudio, torchvision
---
Name: transformers
Version: 4.49.0
Summary: State-of-the-art Machine Learning for JAX, PyTorch and TensorFlow
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: C:\Users\yishu\AppData\Local\Programs\Python\Python313\Lib\site-packages
Requires: filelock, huggingface-hub, numpy, packaging, pyyaml, regex, requests, safetensors, tok

In [ ]:
# Algorithm 1 inserting inside hadamard randomized sampling: Distributed Randomized Regression
# Step 1: Hadamard transform with Rademacher Matrix on Data X
import torch

def hadamard_rademacher(X, n):
    D = torch.randint(2, (n,), dtype=torch.float32) * 2 - 1
    X_rademacher = torch.zeros_like(torch.from_numpy(X))
    for i in range(n):
        x = X[i]
        x_rademacher = torch.zeros_like(torch.from_numpy(x))
        for j in range(n // 2):
            x_rademacher[j] = x[j] + x[j + n // 2]
            x_rademacher[j + n // 2] = x[j] - x[j + n // 2]
        X_rademacher[i] = x_rademacher * D[i]
    return X_rademacher
# Step 2: Uniform sampling on the Hadamard transformed X
def Hadamard_Randomized_Sampling_1(X,Y,n,m,q):
    X = np.array(X)
    Y = np.array(Y)
    x_hat_list = []
    X = np.array(hadamard_rademacher(X,n))
    for k in range(q):
        index = np.random.choice(n, size=m, replace=False)
        X_sk = X[index]
        Y_sk = Y[index]
        L,low = cho_factor(X_sk.T@X_sk)
        x_hat = cho_solve((L,low),X_sk.T@Y_sk)
        x_hat_list.append(x_hat)
    x_bar = sum(x_hat_list) / q
    return x_bar

**Now let us compare these two methods with the optimal solver x_Star**<br>
*We print the optimal norm calculation as an optimal line*<br>
*Then we draw graph of calculated norm against the number of worker (q)*

In [ ]:
import matplotlib.pyplot as plt

# Calculate the norms for each method
norm_uniform = []
norm_hadamard = []
norm_star = []

for q in range(1, 10):  # adjust the range as needed
    x_bar_uniform = uniform_sampling_1(X, Y, n, m, q)
    x_bar_hadamard = Hadamard_Randomized_Sampling_1(X, Y, n, m, q)
    norm_uniform.append(calculate_the_norm_square(X, Y, x_bar_uniform))
    norm_hadamard.append(calculate_the_norm_square(X, Y, x_bar_hadamard))
    norm_star.append(norm_Star)  # optimal solution norm

# Create the plot
plt.plot(range(1, 10), norm_uniform, label='Uniform Sampling')
plt.plot(range(1, 10), norm_hadamard, label='Hadamard Randomized Sampling')
plt.plot(range(1, 10), norm_star, label='Optimal Solution', linestyle='--')

plt.xlabel('Number of Workers (q)')
plt.ylabel('Calculated Norm')
plt.title('Comparison of Sampling Methods')
plt.legend()
plt.show()